# 5a단계: YOLO 파인튜닝용 학습 데이터 만들기

| 순서 | 내용 | 시간 |
|---|---|---|
| ① | 튜닝 클립(0143, 0157)에서 학습용 프레임 40장 고르기 | 자동, 2분 |
| ② | v5 결과(박스 + 팀)를 YOLO 라벨 초안으로 변환 → 미리보기 | 자동 |
| ③ | Roboflow 업로드용 zip 만들기 → 다운로드 | 자동 |
| ④ | **Roboflow에서 초안 고치기** (가이드는 맨 아래) | 직접, 1~1.5시간 |

- 학습 데이터는 **튜닝 클립에서만** 만들어요. 평가 클립(0144, 0156)의 정답 20장은 채점 전용이라 절대 섞지 않아요.
- 클래스는 **Hongik / Kookmin / Referee** 3개예요. YOLO 하나로 탐지와 팀 분류를 같이 하도록 학습할 거예요.

**런타임: CPU로 충분** (GPU 아끼기)

In [ ]:
import os, json, zipfile, shutil
import numpy as np
import cv2
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

import subprocess, matplotlib.font_manager as fm
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run("apt-get -qq install -y fonts-nanum > /dev/null", shell=True)
if os.path.exists(_font):
    fm.fontManager.addfont(_font); plt.rcParams["font.family"] = "NanumGothic"
else:
    print("⚠️ 한글 글꼴 설치 실패: 그래프의 한글이 네모로 보일 수 있어요")
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# ① 설정
FOLDER_PATH      = "/content/drive/MyDrive/2025년 홍익 vs 국민 드론뷰"
DRIVE_OUTPUT_DIR = os.path.join(FOLDER_PATH, "pipeline_outputs")
EVAL_DIR         = os.path.join(DRIVE_OUTPUT_DIR, "eval")
V5_DIR           = os.path.join(DRIVE_OUTPUT_DIR, "v5")          # run_v5 결과 (ROI 따라가기 버전)
FT_DIR           = os.path.join(EVAL_DIR, "finetune")            # 이번 단계 결과
IMG_DIR, PRE_DIR = os.path.join(FT_DIR, "images"), os.path.join(FT_DIR, "prelabels")
for d in (IMG_DIR, PRE_DIR):
    os.makedirs(d, exist_ok=True)

TRAIN_CLIPS = {                       # 학습용 = 튜닝 클립만
    "DJI_20250913133530_0143_D": 20,  # 클립: 뽑을 프레임 수
    "DJI_20250913135111_0157_D": 20,
}
EVAL_CLIPS = ["DJI_20250913133612_0144_D", "DJI_20250913135020_0156_D"]   # 섞이면 안 되는 클립
CLASS_NAMES = ["Hongik", "Kookmin", "Referee"]
MIN_PLAYERS = 12                      # 선수가 너무 적은 프레임(플레이 사이 휴식)은 제외
EDGE_SKIP = 0.03                      # 클립 앞뒤 3%는 제외

assert not set(TRAIN_CLIPS) & set(EVAL_CLIPS), "평가 클립이 학습 데이터에 들어가면 안 됩니다"


def video_path_of(stem):
    for ext in (".MP4", ".mp4", ".MOV", ".mov"):
        p = os.path.join(FOLDER_PATH, stem + ext)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"영상을 찾을 수 없습니다: {stem}")


def load_v5(stem):
    p = os.path.join(V5_DIR, f"{stem}_tracking_final.json")
    assert os.path.exists(p), f"v5 결과가 없습니다: {p}\n→ run_v5_pipeline 노트북을 먼저 실행하세요."
    data = json.load(open(p, encoding="utf-8"))
    return {fr["frame_id"]: fr["players"] for fr in data["frames"]}, data


def pick_frames(by_f, n):
    """선수가 충분한 프레임 중에서 클립 전체에 고르게 n장"""
    fids = sorted(by_f)
    lo, hi = fids[0] + int(len(fids) * EDGE_SKIP), fids[-1] - int(len(fids) * EDGE_SKIP)
    ok = [f for f in fids if lo <= f <= hi and len(by_f[f]) >= MIN_PLAYERS]
    if len(ok) <= n:
        return ok
    return [ok[int(round(i))] for i in np.linspace(0, len(ok) - 1, n)]


def read_frames(stem, wanted):
    """영상을 처음부터 순서대로 읽어 원하는 프레임만 (프레임 번호는 1부터, v3·v5와 같게)"""
    cap = cv2.VideoCapture(video_path_of(stem)); out, fid, last = {}, 0, max(wanted)
    want = set(wanted)
    while fid < last:
        ok, img = cap.read()
        if not ok:
            break
        fid += 1
        if fid in want:
            out[fid] = img
    cap.release()
    return out


def to_yolo_line(cls_id, box, W, H):
    x1, y1, x2, y2 = [float(v) for v in box]
    x1, x2 = max(0, x1), min(W - 1, x2); y1, y2 = max(0, y1), min(H - 1, y2)
    return f"{cls_id} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}"

In [ ]:
# ② 프레임 고르기 + 라벨 초안
NAMES, STATS = [], []
for stem, n in TRAIN_CLIPS.items():
    by_f, _ = load_v5(stem)
    fids = pick_frames(by_f, n)
    imgs = read_frames(stem, fids)
    cnt = {c: 0 for c in CLASS_NAMES}
    for fid in fids:
        img = imgs.get(fid)
        if img is None:
            print(f"⚠️ {stem} {fid}프레임을 읽지 못함"); continue
        H, W = img.shape[:2]
        name = f"{stem}_f{fid:05d}"
        cv2.imwrite(os.path.join(IMG_DIR, name + ".jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
        lines = []
        for p in by_f[fid]:
            if p["type"] in CLASS_NAMES:
                lines.append(to_yolo_line(CLASS_NAMES.index(p["type"]), p["bbox"], W, H)); cnt[p["type"]] += 1
        open(os.path.join(PRE_DIR, name + ".txt"), "w").write("\n".join(lines))
        NAMES.append(name)
    STATS.append({"클립": stem[-6:-2], "프레임": len(fids), "프레임 번호 예": fids[:3] + ["…"] + fids[-2:], **cnt})
    print(f"{stem[-6:-2]}: {len(fids)}장 · 박스 {sum(cnt.values())}개 {cnt}")
print(f"\n✅ 학습용 프레임 {len(NAMES)}장 준비 → {IMG_DIR}")

In [ ]:
# 라벨 초안 미리보기 (파랑=홍익 · 빨강=국민대 · 노랑=심판)
COLORS = {0: (255, 140, 0), 1: (40, 40, 230), 2: (0, 215, 255)}   # BGR


def yolo_to_xyxy(line, W, H):
    c, cx, cy, w, h = line.split()
    cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
    return int(c), (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)


def draw_labels(img, txt):
    H, W = img.shape[:2]; out = img.copy()
    for line in open(txt).read().strip().splitlines():
        c, b = yolo_to_xyxy(line, W, H)
        cv2.rectangle(out, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), COLORS.get(c, (255, 255, 255)), 2)
    return out

for name in NAMES[:: max(1, len(NAMES) // 4)][:4]:
    img = cv2.imread(os.path.join(IMG_DIR, name + ".jpg"))
    plt.figure(figsize=(16, 9)); plt.imshow(cv2.cvtColor(draw_labels(img, os.path.join(PRE_DIR, name + ".txt")), cv2.COLOR_BGR2RGB))
    plt.title(name); plt.axis("off"); plt.show()

In [ ]:
# ③ Roboflow 업로드용 zip (YOLO 형식: images/ + labels/ + data.yaml)
UPLOAD_ZIP = os.path.join(FT_DIR, "finetune_prelabels_yolo.zip")
with zipfile.ZipFile(UPLOAD_ZIP, "w") as z:
    for name in NAMES:
        z.write(os.path.join(IMG_DIR, name + ".jpg"), f"images/{name}.jpg")
        z.write(os.path.join(PRE_DIR, name + ".txt"), f"labels/{name}.txt")
    z.writestr("data.yaml", "names:\n" + "".join(f"  - {n}\n" for n in CLASS_NAMES) + f"nc: {len(CLASS_NAMES)}\n")
print("✅ 업로드용 zip:", UPLOAD_ZIP, f"({len(NAMES)}장)")

from google.colab import files
files.download(UPLOAD_ZIP)

## 🏷️ Roboflow 라벨링 가이드

**1. 업로드**: Roboflow에서 **새 프로젝트**(Object Detection, 이름 예: `drone-football-train`)를 만들고 위 zip을 통째로 드래그하세요. 박스가 그려진 채로 올라와요.
- ⚠️ 예전 평가용(시험지) 프로젝트에 섞어 넣지 마세요.

**2. 수정 규칙** (지난번 정답 20장과 같은 기준)

| 상황 | 할 일 |
|---|---|
| **박스 없는 선수/심판** (특히 엉켜 있는 무리) | **추가** ← 이번에 가장 중요! 파인튜닝으로 줄이고 싶은 게 이 '놓친 선수'예요 |
| 팀이 틀린 박스 | **클래스 수정** |
| 한 사람에 박스 2개 | 하나 **삭제** |
| 두 사람을 감싼 박스 1개 | 하나로 줄이고 나머지 한 명 **추가** |
| 그림자·장비·잔디를 잡은 박스 | **삭제** |
| 경기장 밖 사람 (벤치, 코치, 관중) | **라벨링하지 않음** (초안에 있으면 삭제) |
| 반쯤 가려진 선수 | **보이는 만큼 박스** |

- 박스는 대략 감싸기만 하면 돼요. 몇 px 정밀하게 맞출 필요 없어요.
- 팀 구분이 정말 안 되는 선수는 **가장 그럴듯한 팀**으로 두세요 (Unknown 클래스는 없어요).

**3. 내보내기**: Generate Version → Preprocessing의 **Resize는 끄고**, **Augmentation 없음** → Export → **YOLOv8** 형식 zip 다운로드
- Train/Valid/Test 나누기 비율은 아무렇게나 해도 돼요. 다음 노트북에서 다시 나눠요.

**4.** 받은 zip을 Drive의 `pipeline_outputs/eval/finetune/` 폴더에 **`finetune_gt.zip`** 이름으로 넣으세요. (Windows에서 이름이 `finetune_gt.zip.zip`이 되지 않게 주의)